<a href="https://colab.research.google.com/github/tausif04/CKD-Prediction/blob/main/FYDP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [59]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [60]:
BASE_DIR = "/content/drive/MyDrive/FYDP THESIS"
DATA_PATH = "/content/drive/MyDrive/FYDP THESIS /Dataset/tabulart/chronic_kidney_disease_full.arff"


In [61]:
!pip install -q shap torch torchvision scikit-learn pandas numpy matplotlib seaborn

In [62]:
import pandas as pd
import numpy as np
!pip install -q liac-arff

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer

In [63]:
import io
from scipy.io import arff
import pandas as pd
import numpy as np

# Read the entire ARFF file content
with open(DATA_PATH, 'r') as f:
    arff_content = f.read()

# Pre-process the content to strip whitespace, handle various missing value indicators as '?',
# and lowercase nominals.
lines = arff_content.splitlines()
processed_lines = []
in_data_section = False

# Define a set of all recognized missing value indicators (case-insensitive)
# This includes empty string, single space, hyphen, question mark (as it can appear raw)
# Added 'no' and 'good' to the missing value indicators as they were causing errors for nominal columns.
missing_value_indicators_set = {'', '-', '?', ' ', '\t', '\n', '\r', 'no', 'good'}

for line in lines:
    stripped_line = line.strip()

    if stripped_line.lower() == '@data':
        in_data_section = True
        processed_lines.append(line)
        continue

    if in_data_section:
        # Skip empty lines within the data section to prevent IndexError
        if not stripped_line:
            continue

        cleaned_values = []
        for v in line.split(','): # split by comma
            v_stripped = v.strip().lower() # strip and lowercase immediately for robust comparison

            # Check if the value (after stripping and lowercasing) is considered a missing value
            if v_stripped in missing_value_indicators_set:
                cleaned_values.append('?') # Replace with standard ARFF missing value symbol
            else:
                cleaned_values.append(v_stripped) # Keep other values as lowercased strings

        processed_lines.append(','.join(cleaned_values))
    else:
        processed_lines.append(line)

cleaned_arff_content = '\n'.join(processed_lines)

# Load the cleaned content using io.StringIO
data, meta = arff.loadarff(io.StringIO(cleaned_arff_content))
df = pd.DataFrame(data)

# Convert byte strings to regular strings for object columns
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].apply(lambda x: x.decode('utf-8') if isinstance(x, bytes) else x)

# Replace '?' (which `arff.loadarff` might have converted to string '?' for some types) with np.nan.
# It usually converts to None for nominals, so fillna covers that.
df.replace('?', np.nan, inplace=True)
df.fillna(np.nan, inplace=True) # Converts None values (from '?') to np.nan

# The subsequent manual replacements are redundant if the preprocessing is thorough,
# but keeping them for maximum robustness against anything missed or introduced post-loading.
df.replace("", np.nan, inplace=True)
df.replace("-", np.nan, inplace=True)
df.replace(" ", np.nan, inplace=True)

numeric_cols = [
    "age","bp","bgr","bu","sc","sod",
    "pot","hemo","pcv","wbcc","rbcc" # Corrected 'wc' to 'wbcc' and 'rc' to 'rbcc'
]

categorical_cols = [
    "sg","al","su","rbc","pc","pcc","ba",
    "htn","dm","cad","appet","pe","ane"
]

target_col = "class"
df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors='coerce')

In [64]:
df.head()


,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,...,pcv,wbcc,rbcc,htn,dm,cad,appet,pe,ane,class
0,48.0,80.0,1.020,1,0,NaN,normal,notpresent,notpresent,121.0,...,44.0,7800.0,5.2,yes,yes,NaN,NaN,NaN,NaN,ckd
1,7.0,50.0,1.020,4,0,NaN,normal,notpresent,notpresent,NaN,...,38.0,6000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ckd
2,62.0,80.0,1.010,2,3,normal,normal,notpresent,notpresent,423.0,...,31.0,7500.0,NaN,NaN,yes,NaN,poor,NaN,yes,ckd
3,48.0,70.0,1.005,4,0,normal,abnormal,present,notpresent,117.0,...,32.0,6700.0,3.9,yes,NaN,NaN,poor,yes,yes,ckd
4,51.0,80.0,1.010,2,0,normal,normal,notpresent,notpresent,106.0,...,35.0,7300.0,4.6,NaN,NaN,NaN,NaN,NaN,NaN,ckd


In [65]:
df.isna().sum()


,0
age,9
bp,12
sg,47
al,46
su,49
rbc,152
pc,65
pcc,4
ba,4
bgr,44


In [66]:
# handling missing value (Neumerical)
from sklearn.impute import SimpleImputer

num_imputer = SimpleImputer(strategy="median")

df[numeric_cols] = num_imputer.fit_transform(df[numeric_cols])


In [67]:
# handling missing value (Categorical)
cat_imputer = SimpleImputer(strategy="most_frequent")

df[categorical_cols] = cat_imputer.fit_transform(df[categorical_cols])


In [68]:
df.isna().sum()


,0
age,0
bp,0
sg,0
al,0
su,0
rbc,0
pc,0
pcc,0
ba,0
bgr,0


In [69]:
#Encode Categorical Variables
from sklearn.preprocessing import LabelEncoder
import numpy as np
import pandas as pd # Ensure pandas is imported
from scipy.io import arff # Ensure arff is imported for meta

categorical_cols = [
    "sg","al","su","rbc","pc","pcc","ba",
    "htn","dm","cad","appet","pe","ane"
]

for col in categorical_cols:
    df[col] = LabelEncoder().fit_transform(df[col])

# --- BEGIN FIX FOR 'class' COLUMN INTEGRITY ---
# Regenerate 'class' column from raw data to ensure its integrity,
# as its state appears corrupted before this cell's execution.
# This assumes 'data' and 'meta' variables (from arff.loadarff) are still available and correct.

# Safely get the 'class' column index from metadata
class_col_index = meta.names().index('class')
original_class_data_series = pd.Series([row[class_col_index] for row in data])

# Apply the same byte decoding and missing value handling as in 9l2-NO8Zxv6b for the class column
df['class'] = original_class_data_series.apply(lambda x: x.decode('utf-8') if isinstance(x, bytes) else x)
df['class'].replace('?', np.nan, inplace=True)
df['class'].fillna(np.nan, inplace=True)
# --- END FIX FOR 'class' COLUMN INTEGRITY ---

#Encode Target Variable more robustly
# Ensure target column values are consistently strings, stripped, and lowercased before mapping
df['class'] = df['class'].astype(str).str.strip().str.lower()

# Debug: Check unique values in 'class' before mapping
print("Unique values in 'class' before mapping:")
print(df['class'].value_counts(dropna=False))

df["class"] = df["class"].map({"ckd": 1, "notckd": 0})
# Convert back any 'nan' strings (which result from np.nan.astype(str)) to actual np.nan
df['class'] = df['class'].replace('nan', np.nan)


Unique values in 'class' before mapping:
class
ckd       250
notckd    149
nan         1
Name: count, dtype: int64


/tmp/ipython-input-2398172921.py:26: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['class'].replace('?', np.nan, inplace=True)
/tmp/ipython-input-2398172921.py:27: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try us

In [70]:
#Train test split
from sklearn.model_selection import train_test_split

# Drop rows where 'class' is NaN
df_cleaned = df.dropna(subset=['class'])

X = df_cleaned.drop(columns=["class"])
y = df_cleaned["class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [71]:
#Convert Data to PyTorch Tensors
import torch
from torch.utils.data import TensorDataset, DataLoader

X_train_tensor = torch.tensor(X_train.values, dtype=torch.float32)
X_test_tensor  = torch.tensor(X_test.values, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32)
y_test_tensor  = torch.tensor(y_test.values, dtype=torch.float32)


In [72]:
#Create DataLoaders
batch_size = 32

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset  = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)


In [73]:
# Define the MLP Model (Tabular Branch)
import torch.nn as nn

class TabularMLP(nn.Module):
    def __init__(self, input_dim):
        super(TabularMLP, self).__init__()

        self.model = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.model(x)


In [74]:
import torch.nn as nn
import random
import os

# Set random seeds for reproducibility
def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)

set_seed(42)

# Initialize Model, Loss, Optimizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = TabularMLP(input_dim=X_train.shape[1]).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [75]:
# Training Loop
epochs = 50

for epoch in range(epochs):
    model.train()
    epoch_loss = 0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device).unsqueeze(1)

        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {epoch_loss:.4f}")


Epoch [10/50], Loss: 4.9854
Epoch [20/50], Loss: 3.1615
Epoch [30/50], Loss: 2.0988
Epoch [40/50], Loss: 1.9005
Epoch [50/50], Loss: 1.6274


In [76]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

model.eval()
y_true, y_pred = [], []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        outputs = model(X_batch)
        probs = torch.sigmoid(outputs)
        preds = (probs > 0.5).int().cpu().numpy()

        y_pred.extend(preds.flatten())
        y_true.extend(y_batch.numpy())

print("Accuracy :", accuracy_score(y_true, y_pred))
print("Precision:", precision_score(y_true, y_pred))
print("Recall   :", recall_score(y_true, y_pred))
print("F1-score :", f1_score(y_true, y_pred))


Accuracy : 0.825
Precision: 1.0
Recall   : 0.72
F1-score : 0.8372093023255814


In [78]:
!pip install -q xgboost
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

xgb = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42
)

xgb.fit(X_train, y_train)

y_pred = xgb.predict(X_test)

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1-score :", f1_score(y_test, y_pred))




Accuracy : 1.0
Precision: 1.0
Recall   : 1.0
F1-score : 1.0
